# Bug Tests

Regression tests for known bugs in simplinho, compared against PuLP/CBC.

In [ ]:
import sys
import numpy as np
import pulp

# Load simplinho from local build
for path in ['build-local', 'build', 'build-verify']:
    try:
        sys.path.insert(0, path)
        import simplinho as splx
        print(f'Loaded simplinho from {path}')
        break
    except ImportError:
        sys.path.pop(0)

def pulp_solve(c, A_ub, b_ub, A_eq, b_eq, l, u):
    """Solve min c^T x with A_ub x <= b_ub, A_eq x = b_eq, l <= x <= u via PuLP/CBC."""
    n = len(c)
    prob = pulp.LpProblem('bug_test', pulp.LpMinimize)
    xs = [pulp.LpVariable(f'x{j}', lowBound=l[j], upBound=None if u[j] >= 1e29 else u[j])
          for j in range(n)]
    prob += pulp.lpSum(c[j] * xs[j] for j in range(n))
    if A_ub is not None:
        for i, row in enumerate(A_ub):
            prob += pulp.lpSum(row[j] * xs[j] for j in range(n)) <= b_ub[i]
    if A_eq is not None:
        for i, row in enumerate(A_eq):
            prob += pulp.lpSum(row[j] * xs[j] for j in range(n)) == b_eq[i]
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return {
        'status': pulp.LpStatus[prob.status],
        'obj': pulp.value(prob.objective),
        'x': [pulp.value(xs[j]) for j in range(n)],
    }

def compare(name, splx_sol, pulp_result, tol=1e-5):
    splx_ok = str(splx_sol.status) == 'LPStatus.Optimal'
    pulp_ok  = pulp_result['status'] == 'Optimal'
    obj_match = splx_ok and pulp_ok and abs(splx_sol.obj - pulp_result['obj']) < tol
    x_match   = splx_ok and pulp_ok and np.allclose(splx_sol.x, pulp_result['x'], atol=tol)
    passed = splx_ok and pulp_ok and obj_match
    tag = '✓ PASS' if passed else '✗ FAIL'
    print(f'--- {name} {tag} ---')
    print(f'  simplinho: status={splx_sol.status}  obj={splx_sol.obj:.6g}  x={np.round(splx_sol.x, 6)}')
    print(f'  pulp/CBC:  status={pulp_result["status"]}  obj={pulp_result["obj"]:.6g}  x={np.round(pulp_result["x"], 6)}')
    if not passed:
        if not splx_ok:
            print(f'  ISSUE: simplinho returned {splx_sol.status} (expected Optimal)')
        elif not obj_match:
            print(f'  ISSUE: obj mismatch: {splx_sol.obj:.10g} vs {pulp_result["obj"]:.10g}')
    print()

## BUG-000: bounded_box (fixed)

Single equality constraint with all variables doubly bounded.
Was returning `Singular` — fixed by constructing a valid logical basis in the reformulation path.

In [ ]:
# min -x0 + 0.5*x1 + 2*x2
# s.t.  x0 - x1 + x2 = 2
#        0 <= xi <= 3
# Optimal: x=[3,1,0], obj=-2.5

A_eq = np.array([[1.0, -1.0, 1.0]])
b_eq = np.array([2.0])
c    = np.array([-1.0, 0.5, 2.0])
l    = np.zeros(3)
u    = np.full(3, 3.0)

solver   = splx.RevisedSimplex()
splx_sol = solver.solve(A_eq, b_eq, c, l, u)
pulp_res = pulp_solve(c, None, None, A_eq, b_eq, l, u)

compare('BUG-000: bounded_box', splx_sol, pulp_res)

## BUG-001: doubly-bounded multi-row LP returns Singular (open)

Two constraint rows, all variables with finite upper bounds.
The sparse bound-reformulation path maps the warm-start basis to the 4×4 reformulated
system; the mapped basis passes the primal+dual feasibility check but the inner primal
simplex still returns Singular.

**Status:** open — `map_reformulated_basis_state_` likely has an index/sign error
when `m_eq > 1` and all variables are doubly-bounded.

In [ ]:
# min -3*x0 - 5*x1
# s.t.   x0 + 2*x1 <= 12
#        2*x0 +  x1 <= 10
#        0 <= xi <= 10
# Optimal: x=[2.667, 4.667], obj=-27.333

# Reformulate to equality form: add slacks s0, s1
A_eq = np.array([[1.0, 2.0, 1.0, 0.0],
                 [2.0, 1.0, 0.0, 1.0]])
b_eq = np.array([12.0, 10.0])
c    = np.array([-3.0, -5.0, 0.0, 0.0])
l    = np.zeros(4)
u    = np.array([10.0, 10.0, np.inf, np.inf])

solver   = splx.RevisedSimplex()
splx_sol = solver.solve(A_eq, b_eq, c, l, u)

# PuLP comparison (original inequality form)
A_ub = np.array([[1.0, 2.0], [2.0, 1.0]])
b_ub = np.array([12.0, 10.0])
c2   = np.array([-3.0, -5.0])
l2   = np.zeros(2)
u2   = np.full(2, 10.0)
pulp_res = pulp_solve(c2, A_ub, b_ub, None, None, l2, u2)

# Adapt splx solution back to 2-var space for comparison
import copy
splx_2var = copy.copy(splx_sol)
splx_2var.x   = splx_sol.x[:2] if splx_sol.x is not None and len(splx_sol.x) >= 2 else splx_sol.x
splx_2var.obj = float(c2 @ splx_2var.x) if str(splx_sol.status) == 'LPStatus.Optimal' else float('nan')

compare('BUG-001: doubly-bounded multi-row', splx_2var, pulp_res)

print('Debug info (simplinho):')
print('  reformulation info:', {k: v for k, v in splx_sol.info.items() if 'reform' in k or 'basis' in k})

### Workaround for BUG-001

Add upper bounds as explicit inequality rows instead of passing finite `u`.

In [ ]:
# Workaround: convert x <= u to explicit equality rows with slacks
# Original: A_eq x = b, 0 <= x <= u
# Expanded: [A_eq  0 ] [x]   [b]
#           [I     I ] [s] = [u],  x,s >= 0, u_expanded = inf

n_orig = 2
u_orig = np.array([10.0, 10.0])

A_base = np.array([[1.0, 2.0, 1.0, 0.0],
                   [2.0, 1.0, 0.0, 1.0]])
b_base = np.array([12.0, 10.0])
c_base = np.array([-3.0, -5.0, 0.0, 0.0])
l_base = np.zeros(4)
u_base = np.full(4, np.inf)

# Add upper-bound rows: x0 + s2 = 10, x1 + s3 = 10
ub_rows = np.zeros((n_orig, 4 + n_orig))
for i in range(n_orig):
    ub_rows[i, i] = 1.0          # xi
    ub_rows[i, 4 + i] = 1.0      # slack si

A_aug = np.hstack([np.vstack([A_base, np.zeros((n_orig, 4))]),
                   np.vstack([np.zeros((2, n_orig)), np.eye(n_orig)])])
# Simpler manual construction:
A_aug = np.array([[1,2,1,0,0,0],
                  [2,1,0,1,0,0],
                  [1,0,0,0,1,0],
                  [0,1,0,0,0,1]], dtype=float)
b_aug = np.array([12.0, 10.0, 10.0, 10.0])
c_aug = np.array([-3.0, -5.0, 0.0, 0.0, 0.0, 0.0])
l_aug = np.zeros(6)
u_aug = np.full(6, np.inf)

solver     = splx.RevisedSimplex()
splx_wk    = solver.solve(A_aug, b_aug, c_aug, l_aug, u_aug)
pulp_res   = pulp_solve(c2, A_ub, b_ub, None, None, l2, u2)

splx_wk_2  = copy.copy(splx_wk)
splx_wk_2.x   = splx_wk.x[:2]
splx_wk_2.obj = float(c2 @ splx_wk_2.x) if str(splx_wk.status) == 'LPStatus.Optimal' else float('nan')

compare('BUG-001 workaround: explicit ub rows', splx_wk_2, pulp_res)